In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    image_feex_path: Path
    main_model_path: Path
    vectorizer_path: Path
    images_dir: Path
    trained_model_path: Path
    train_images_captions_path: Path

    SPLIT_SIZE: int
    RANDOM_STATE: int         
    MAX_LENGTH: int
    BATCH_SIZE: int          
    

In [6]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories,load_pkl_file

In [7]:
import pickle
from imgCaption import logger
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img,img_to_array

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        training = self.config.model_trainer
        prepare_base_model = self.config.prepare_base_model
        data_transformation = self.config.data_transformation
        params = self.params

        create_directories([training.root_dir])

        prepare_model_trainer_config = ModelTrainerConfig(
                                            root_dir=Path(training.root_dir),
                                            image_feex_path=Path(prepare_base_model.image_feex_path),
                                            main_model_path=Path(prepare_base_model.main_model_path),
                                            vectorizer_path=Path(data_transformation.vectorizer_path),
                                            images_dir=Path(training.images_dir),
                                            trained_model_path=Path(training.trained_model_path),
                                            train_images_captions_path=Path(data_transformation.train_images_captions_path),
                                            SPLIT_SIZE=params.SPLIT_SIZE,
                                            RANDOM_STATE=params.RANDOM_STATE,
                                            MAX_LENGTH=params.MAX_LENGTH,
                                            BATCH_SIZE=params.BATCH_SIZE
                                        )

        return prepare_model_trainer_config

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.load_vectorizer()
        self.load_feature_extractor()
        self.load_main_model()

    def load_vectorizer(self):
        from_disk=load_pkl_file(self.config.vectorizer_path)
        self.vectorizer = TextVectorization.from_config(from_disk["config"])
        self.vectorizer.adapt(tf.data.Dataset.from_tensor_slices(["dummy"]))
        self.vectorizer.set_weights(from_disk["weights"])

    def load_feature_extractor(self):
        self.feature_extractor = load_model(self.config.image_feex_path)

    def load_main_model(self):
        self.main_model = load_model(self.config.main_model_path)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
        
    def split_dict_by_keys(self,image_ids):
        train,val=train_test_split(image_ids,test_size=self.config.SPLIT_SIZE,random_state=self.config.RANDOM_STATE)
        logger.info(f"Length of train_images is : {len(train)}")
        logger.info(f"Length of val_images is : {len(val)}")
        return train,val
        
    def features_ext_dict(self,image_caption_map):
        d={}
        images=list(image_caption_map.keys())
        batch_size = 32

        for start in tqdm(range(0, len(images), batch_size), desc="Extracting features", unit="batch"):
            batch_names = images[start : start + batch_size]
            batch_imgs = []
            for img_name in batch_names:
                img_path = os.path.join(self.config.images_dir, img_name)
                img = load_img(img_path, target_size=(224, 224))
                img = img_to_array(img)
                img = preprocess_input(img)
                batch_imgs.append(img) 
            batch_imgs = np.array(batch_imgs)
            features = self.feature_extractor.predict(batch_imgs, verbose=0)
            for i, image in enumerate(batch_names):
                d[image] = features[i].flatten()                
            del batch_imgs, features
        logger.info(f"Feature extraction complete: {len(d)} images processed")
        return d
    
    def data_generator(self,image_caption_map,features_map):
        while True:
            X1,X2,Y = list(),list(),list()
            cnt=0
            for image,captions in image_caption_map.items():
                for cap in captions:
                    cap=self.vectorizer([cap]).numpy()[0]
                    for j in range(1,len(cap)):
                        cur_seq=np.array([cap[:j]])
                        cur_seq = pad_sequences(cur_seq, maxlen=self.config.MAX_LENGTH, padding='post')[0]
                        next_word=cap[j]
                        X1.append(features_map[image])
                        X2.append(cur_seq)
                        Y.append(next_word)
                cnt+=1
                if cnt==self.config.BATCH_SIZE:
                    yield [np.array(X1),np.array(X2)],np.array(Y)
                    X1,X2,Y = list(),list(),list()
                    cnt=0   
            if len(X1) > 0:
                yield [np.array(X1), np.array(X2)], np.array(Y)

    def train(self):
        train_images_caption=load_pkl_file(self.config.train_images_captions_path)
        image_ids = list(train_images_caption.keys())
        train_keys, val_keys = self.split_dict_by_keys(image_ids)

        train_img_cap_map = {k: train_images_caption[k] for k in train_keys}
        val_img_cap_map   = {k: train_images_caption[k] for k in val_keys}
        train_features_map = self.features_ext_dict(train_images_caption)

        train_feat_map = {k: train_features_map[k] for k in train_keys}
        val_feat_map   = {k: train_features_map[k] for k in val_keys}

        train_generator = self.data_generator(train_img_cap_map, train_feat_map)
        val_generator = self.data_generator(val_img_cap_map, val_feat_map)


        self.main_model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        logger.info("Model Training Started")
        steps_per_epoch = len(train_img_cap_map) // self.config.BATCH_SIZE
        validation_steps = len(val_img_cap_map) // self.config.BATCH_SIZE
        self.main_model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=30,
            steps_per_epoch=steps_per_epoch,
            validation_steps=validation_steps
        )  
        self.save_model(
        path=self.config.trained_model_path,
        model=self.main_model
        ) 

In [10]:
try:
    config=ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
    
except Exception as e:
    raise e

[2026-07-06 16:59:29,795: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-06 16:59:29,800: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-06 16:59:29,802: INFO: common: created directory at: artifacts]
[2026-07-06 16:59:29,803: INFO: common: created directory at: artifacts/training]
[2026-07-06 16:59:31,869: WARNING: hdf5_format: No training configuration found in the save file, so the model was *not* compiled. Compile it manually.]
[2026-07-06 16:59:34,166: WARNING: hdf5_format: No training configuration found in the save file, so the model was *not* compiled. Compile it manually.]
[2026-07-06 16:59:34,180: INFO: 1578035137: Length of train_images is : 5177]
[2026-07-06 16:59:34,180: INFO: 1578035137: Length of val_images is : 1295]


Extracting features: 100%|██████████| 203/203 [08:23<00:00,  2.48s/batch]

[2026-07-06 17:07:57,241: INFO: 1578035137: Feature extraction complete: 6472 images processed]
[2026-07-06 17:07:57,273: INFO: 1578035137: Model Training Started]


Epoch 1/30
 14/323 [>.............................] - ETA: 48:11 - loss: 7.0280 - accuracy: 0.0899

KeyboardInterrupt: 